## Presupuesto y Empaquetado


In [1]:
import time
from typing import List, Dict, Any


def calculate_sentence_size(sentence: str) -> int:
    """
    Calcula el tamaño de una oración.
    
    Bajo el supuesto del proveedor, 1 carácter equivale a 1 unidad de texto.
    Se eliminan los espacios en blanco sobrantes en los extremos para un cálculo exacto.
    """
    return len(sentence.strip())


def build_dynamic_batches(sentences: List[str], max_units: int) -> List[List[str]]:
    """
    Agrupa dinámicamente las oraciones en lotes (batches) optimizados.
    
    Garantiza que la suma de unidades de las oraciones en un lote no exceda `max_units`.
    Si una sola oración excede el límite por sí misma, se maneja como un lote individual 
    o podría generar una excepción según las políticas del backend.
    """
    batches = []
    current_batch = []
    current_batch_size = 0

    for sentence in sentences:
        sentence_size = calculate_sentence_size(sentence)
        
        # Validación de seguridad: si una oración sola supera el máximo permitido
        if sentence_size > max_units:
            print(f"[⚠️ WARNING] La oración excede el límite máximo por sí sola ({sentence_size} unidades). Se enviará en un lote aislado.")
            if current_batch:
                batches.append(current_batch)
                current_batch = []
                current_batch_size = 0
            batches.append([sentence])
            continue

        # Si la oración actual desborda el lote, cerramos el lote actual e iniciamos uno nuevo
        if current_batch_size + sentence_size > max_units:
            batches.append(current_batch)
            current_batch = [sentence]
            current_batch_size = sentence_size
        else:
            # Añadir oración al lote actual
            current_batch.append(sentence)
            current_batch_size += sentence_size

    # Asegurar el envío del último lote si quedó con elementos
    if current_batch:
        batches.append(current_batch)

    return batches


def mock_api_translate(batch: List[str]) -> int:
    """
    Simula el envío de un lote de oraciones a la API de traducción de un tercero.
    
    Introduce un retardo controlado para simular latencia de red y mitigar Rate Limits.
    Devuelve el tamaño total procesado en el lote.
    """
    batch_size = sum(calculate_sentence_size(s) for s in batch)
    
    print(f"[🚀 API] Enviando lote con {len(batch)} oraciones ({batch_size} unidades)...")
    
    # Simulación de latencia de red (I/O Bound)
    time.sleep(0.1) 
    
    return batch_size


def execute_pipeline(sentences: List[str], max_units_per_batch: int) -> Dict[str, Any]:
    """
    Orquesta el flujo completo de empaquetado, simulación de envío y recolección de métricas.
    """
    print("=== Iniciando Proceso de Optimización de Lotes ===")
    
    # 1. Agrupamiento dinámico
    batches = build_dynamic_batches(sentences, max_units_per_batch)
    
    batch_sizes_breakdown = []
    total_units_consumed = 0
    
    # 2. Envío secuencial de lotes
    for index, batch in enumerate(batches, start=1):
        print(f"\nProcesando lote {index}/{len(batches)}...")
        units_processed = mock_api_translate(batch)
        
        # Registro de métricas por lote
        batch_sizes_breakdown.append(units_processed)
        total_units_consumed += units_processed

    print("\n=== Proceso Finalizado con Éxito ===\n")
    
    # 3. Retorno de métricas para el recibo de ejecución
    return {
        "total_sentences": len(sentences),
        "total_units": total_units_consumed,
        "total_requests": len(batches),
        "breakdown": batch_sizes_breakdown
    }


def print_execution_receipt(metrics: Dict[str, Any]) -> None:
    """
    Imprime un recibo detallado con el rendimiento y costos operativos simulados.
    """
    print("=" * 50)
    print("               RECIBO DE EJECUCIÓN              ")
    print("=" * 50)
    print(f"Total de oraciones procesadas : {metrics['total_sentences']}")
    print(f"Total de unidades (caracteres): {metrics['total_units']} unidades")
    print(f"Número total de peticiones   : {metrics['total_requests']} API Requests")
    print("-" * 50)
    print("Desglose de tamaño por lote:")
    for i, size in enumerate(metrics['breakdown'], start=1):
        print(f"  • Lote #{i}: {size} unidades")
    print("=" * 50)


# =====================================================================
# CONFIGURACIÓN Y DATOS DE ENTRADA
# =====================================================================
if __name__ == "__main__":
    # Umbral máximo configurable de unidades por lote (Límite del proveedor)
    MAX_UNITS_PER_BATCH = 250

    # Generación de una lista de 100 oraciones de prueba con longitudes variables
    base_sentences = [
        "El backend debe ser altamente escalable.",
        "Optimizar el uso de tokens reduce drásticamente el costo operativo en la nube.",
        "Monitorear los rate limits evita bloqueos HTTP 429 de proveedores externos.",
        "La arquitectura limpia facilita el mantenimiento del código a largo plazo.",
        "Python es una excelente opción para scripts de automatización de datos.",
        "El buffering de peticiones minimiza la sobrecarga de conexiones I/O."
    ]
    
    # Multiplicamos la base para obtener exactamente 100 oraciones con variación de datos
    input_sentences = []
    for idx in range(100):
        base_str = base_sentences[idx % len(base_sentences)]
        # Añadimos un sufijo dinámico para romper la homogeneidad de caracteres
        input_sentences.append(f"[{idx + 1}] {base_str}")

    # Ejecución del pipeline
    execution_metrics = execute_pipeline(input_sentences, MAX_UNITS_PER_BATCH)
    
    # Salida del reporte financiero/operativo
    print_execution_receipt(execution_metrics)


=== Iniciando Proceso de Optimización de Lotes ===

Procesando lote 1/34...
[🚀 API] Enviando lote con 3 oraciones (205 unidades)...

Procesando lote 2/34...
[🚀 API] Enviando lote con 3 oraciones (225 unidades)...

Procesando lote 3/34...
[🚀 API] Enviando lote con 3 oraciones (205 unidades)...

Procesando lote 4/34...
[🚀 API] Enviando lote con 3 oraciones (228 unidades)...

Procesando lote 5/34...
[🚀 API] Enviando lote con 3 oraciones (208 unidades)...

Procesando lote 6/34...
[🚀 API] Enviando lote con 3 oraciones (228 unidades)...

Procesando lote 7/34...
[🚀 API] Enviando lote con 3 oraciones (208 unidades)...

Procesando lote 8/34...
[🚀 API] Enviando lote con 3 oraciones (228 unidades)...

Procesando lote 9/34...
[🚀 API] Enviando lote con 3 oraciones (208 unidades)...

Procesando lote 10/34...
[🚀 API] Enviando lote con 3 oraciones (228 unidades)...

Procesando lote 11/34...
[🚀 API] Enviando lote con 3 oraciones (208 unidades)...

Procesando lote 12/34...
[🚀 API] Enviando lote con 3 or

In [ ]:
import time
from typing import List, Dict, Any
import requests
from bs4 import BeautifulSoup


def scrape_quotes_from_web() -> List[str]:
    """
    Realiza web scraping en quotes.toscrape.com para extraer las citas de la primera página.
    Aplica buenas prácticas de backend agregando un User-Agent.
    """
    url = "https://quotes.toscrape.com/"
    headers = {
        "User-Agent": "BackendTokenOptimizer/1.0 (Resource Optimization Script)"
    }
    
    print(f"[🌐 SCRAPER] Conectando a {url}...")
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Valida errores HTTP (4xx, 5xx)
    except requests.exceptions.RequestException as e:
        print(f"[❌ ERROR] Fallo al conectar con el sitio web: {e}")
        return []

    # Parsear el HTML con BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Extraer el texto de los elementos <span class="text">
    quotes_elements = soup.find_all("span", class_="text")
    
    # Limpiar y extraer solo el texto interno
    # Las citas vienen envueltas en comillas tipográficas (“...”), las limpiamos para exactitud
    quotes = [q.get_text().strip("“” ") for q in quotes_elements]
    
    print(f"[🎰 SCRAPER] Extracción exitosa: {len(quotes)} citas recuperadas de la web.\n")
    return quotes


def calculate_sentence_size(sentence: str) -> int:
    """
    Calcula el tamaño de una oración.
    
    Asume la regla de negocio: 1 carácter = 1 unidad de texto.
    """
    return len(sentence.strip())


def build_dynamic_batches(sentences: List[str], max_units: int) -> List[List[str]]:
    """
    Algoritmo Greedy / Dinámico para empaquetar oraciones en lotes optimizados.
    Garantiza que la suma de unidades de un lote no exceda `max_units`.
    """
    batches = []
    current_batch = []
    current_batch_size = 0

    for sentence in sentences:
        sentence_size = calculate_sentence_size(sentence)
        
        # Validación de seguridad: tamaño de oración individual
        if sentence_size > max_units:
            print(f"[⚠️ WARNING] Cita muy larga ({sentence_size} uds). Se enviará en lote aislado.")
            if current_batch:
                batches.append(current_batch)
                current_batch = []
                current_batch_size = 0
            batches.append([sentence])
            continue

        # Si desborda el límite, cierra el lote actual e inicia el siguiente
        if current_batch_size + sentence_size > max_units:
            batches.append(current_batch)
            current_batch = [sentence]
            current_batch_size = sentence_size
        else:
            current_batch.append(sentence)
            current_batch_size += sentence_size

    if current_batch:
        batches.append(current_batch)

    return batches


def mock_api_translate(batch: List[str]) -> int:
    """
    Simula el envío del lote a una API de traducción con cobro por tokens.
    Implementa un retraso controlado para mitigar Rate Limits del proveedor.
    """
    batch_size = sum(calculate_sentence_size(s) for s in batch)
    
    print(f"[🚀 API] Enviando lote con {len(batch)} citas ({batch_size} unidades/caracteres)...")
    time.sleep(0.1)  # Simulación de latencia de red
    
    return batch_size


def execute_pipeline(max_units_per_batch: int) -> Dict[str, Any]:
    """
    Orquestador principal: Extrae de la web, agrupa dinámicamente y procesa.
    """
    # 1. Extracción de datos reales mediante Scraping
    input_quotes = scrape_quotes_from_web()
    
    if not input_quotes:
        print("[❌ ERROR] No se encontraron datos para procesar. Abortando pipeline.")
        return {}

    print("=== Iniciando Proceso de Optimización de Lotes ===")
    
    # 2. Empaquetado dinámico
    batches = build_dynamic_batches(input_quotes, max_units_per_batch)
    
    batch_sizes_breakdown = []
    total_units_consumed = 0
    
    # 3. Consumo de la API simulada lote por lote
    for index, batch in enumerate(batches, start=1):
        print(f"\nProcesando lote {index}/{len(batches)}...")
        units_processed = mock_api_translate(batch)
        
        batch_sizes_breakdown.append(units_processed)
        total_units_consumed += units_processed

    print("\n=== Proceso Finalizado con Éxito ===\n")
    
    return {
        "total_sentences": len(input_quotes),
        "total_units": total_units_consumed,
        "total_requests": len(batches),
        "breakdown": batch_sizes_breakdown
    }


def print_execution_receipt(metrics: Dict[str, Any]) -> None:
    """
    Imprime el desglose final de rendimiento y costos operativos en consola.
    """
    if not metrics:
        return
        
    print("=" * 50)
    print("               RECIBO DE EJECUCIÓN              ")
    print("=" * 50)
    print(f"Total de citas procesadas     : {metrics['total_sentences']}")
    print(f"Total de unidades consumidas  : {metrics['total_units']} caracteres")
    print(f"Número total de peticiones API: {metrics['total_requests']} Requests")
    print("-" * 50)
    print("Desglose de tamaño por lote:")
    for i, size in enumerate(metrics['breakdown'], start=1):
        print(f"  • Lote #{i}: {size} unidades")
    print("=" * 50)


# =====================================================================
# CONFIGURACIÓN DEL SISTEMA BACKEND
# =====================================================================
if __name__ == "__main__":
    # Ajustamos el límite máximo por lote. 
    # Las citas de esta web suelen medir entre 50 y 250 caracteres cada una.
    MAX_UNITS_PER_BATCH = 300 

    # Ejecución del pipeline de datos distribuido
    execution_metrics = execute_pipeline(MAX_UNITS_PER_BATCH)
    
    # Despliegue del reporte operativo
    print_execution_receipt(execution_metrics)


[🌐 SCRAPER] Conectando a https://quotes.toscrape.com/...
[🎰 SCRAPER] Extracción exitosa: 10 citas recuperadas de la web.

=== Iniciando Proceso de Optimización de Lotes ===

Procesando lote 1/4...
[🚀 API] Enviando lote con 2 citas (196 unidades/caracteres)...

Procesando lote 2/4...
[🚀 API] Enviando lote con 2 citas (231 unidades/caracteres)...

Procesando lote 3/4...
[🚀 API] Enviando lote con 3 citas (254 unidades/caracteres)...

Procesando lote 4/4...
[🚀 API] Enviando lote con 3 citas (194 unidades/caracteres)...

=== Proceso Finalizado con Éxito ===

               RECIBO DE EJECUCIÓN              
Total de citas procesadas     : 10
Total de unidades consumidas  : 875 caracteres
Número total de peticiones API: 4 Requests
--------------------------------------------------
Desglose de tamaño por lote:
  • Lote #1: 196 unidades
  • Lote #2: 231 unidades
  • Lote #3: 254 unidades
  • Lote #4: 194 unidades
